# Why did it say that? Hands-on with Grad-CAM
AICTE QIP · IIIT Kottayam · FACTS-H Lab

You will ask a real image model **why** it gave its answer, and then **test** that explanation.

**How to work:** press the ▶ button on each grey box, one at a time, from top to bottom. You only choose options from the menus; you do not need to edit any code.

The model: an image model (MobileNetV2) trained on lab photos of tomato leaves. It knows 4 conditions: bacterial spot, early blight, healthy, septoria leaf spot. It has already been trained and saved, so everyone uses the same model and nothing is trained here.

At the end, **Step 3** saves your result as a picture you can keep.

In [ ]:
#@title Step 0 · Setup: press ▶ and wait about one minute

import warnings; warnings.filterwarnings("ignore")
import os, urllib.request, zipfile
def fetch(url, dest):
    if not os.path.exists(dest) or os.path.getsize(dest) == 0:
        print("Downloading", dest, "...")
        urllib.request.urlretrieve(url, dest)
fetch("https://github.com/ansiths-iiitk/leaf-demo/raw/main/leafdata.zip", "leafdata.zip")
if not os.path.exists("leafdata/plantvillage/test"):
    zipfile.ZipFile("leafdata.zip").extractall(".")
for f in ["leaf_head.npz", "reference.json"]:
    fetch(f"https://raw.githubusercontent.com/ansiths-iiitk/xai-demo/main/{f}", f)
assert os.path.exists("leafdata/field/healthy/healthy_field_005.jpg"), "The leaf photos did not download. Run this cell again."
print("Photos and model files downloaded.")

import os, json, numpy as np, tensorflow as tf, matplotlib.pyplot as plt
from PIL import Image
IMG = 224
CLASSES = ["bacterial_spot", "early_blight", "healthy", "septoria_leaf_spot"]
NAMES = ["Bacterial spot", "Early blight", "Healthy", "Septoria leaf spot"]
DEFAULTS = {   # the six photos participants can pick
    "Lab 1 (early blight)":  "leafdata/plantvillage/test/early_blight/early_blight_010.jpg",
    "Lab 2 (healthy)":       "leafdata/plantvillage/test/healthy/healthy_010.jpg",
    "Lab 3 (septoria)":      "leafdata/plantvillage/test/septoria_leaf_spot/septoria_leaf_spot_010.jpg",
    "Field 1 (healthy)":     "leafdata/field/healthy/healthy_field_005.jpg",
    "Field 2 (early blight)":"leafdata/field/early_blight/early_blight_field_005.jpg",
    "Field 3 (septoria)":    "leafdata/field/septoria_leaf_spot/septoria_leaf_spot_field_006.jpg",
}
pre = tf.keras.applications.mobilenet_v2.preprocess_input
base = tf.keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
base.trainable = False
gap = tf.keras.layers.GlobalAveragePooling2D()

def load_photo(path):
    """Same loader everywhere: centre square crop, 224 x 224, RGB, float32 0-255."""
    im = Image.open(path).convert("RGB")
    w, h = im.size; s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((IMG, IMG), Image.BILINEAR)
    return np.asarray(im, dtype="float32")

def predict(img):
    x = tf.convert_to_tensor(img[None])      # a tensor copy: preprocess_input would change a NumPy array in place
    return head(gap(base(pre(x), training=False))).numpy()[0]

def gradcam(img, cls=None):
    x = tf.convert_to_tensor(img[None])
    with tf.GradientTape() as tape:
        conv = base(pre(x), training=False)
        tape.watch(conv)
        probs = head(gap(conv))
        c = int(tf.argmax(probs[0])) if cls is None else cls
        score = probs[:, c]
    g = tape.gradient(score, conv)[0]
    w = tf.reduce_mean(g, axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(conv[0] * w, axis=-1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return tf.image.resize(cam[..., None], (IMG, IMG), method="bilinear").numpy()[..., 0], c

W = np.load("leaf_head.npz")
head = tf.keras.layers.Dense(4, activation="softmax"); head.build((None, 1280)); head.set_weights([W["kernel"], W["bias"]])
ref = json.load(open("reference.json"))
diff = max(float(np.abs(predict(load_photo(p)) - np.array(ref["defaults"][n])).max()) for n, p in DEFAULTS.items())
print("Model ready." if diff < 0.01 else f"Warning: model differs from the class model (difference {diff:.3f}). Tell the instructor.")
print(f"It learned 4 tomato-leaf conditions: {', '.join(NAMES)}.")

plt.rcParams.update({"font.family": "serif", "font.size": 12})
def show_row(images, titles, colours=None):
    fig, axs = plt.subplots(1, len(images), figsize=(4 * len(images), 4.3))
    for i, (ax, im, t) in enumerate(zip(np.atleast_1d(axs), images, titles)):
        ax.imshow(im); ax.set_title(t, fontsize=13, color=(colours[i] if colours else "black")); ax.axis("off")
    plt.tight_layout(); plt.show()
def overlay(img, cam):
    return (plt.cm.jet(cam)[..., :3] * 0.45 + img / 255 * 0.55).clip(0, 1)
def bars(p, title):
    fig, ax = plt.subplots(figsize=(6, 2.2)); ax.barh(NAMES[::-1], p[::-1] * 100, color="#7a1f1f")
    for i, v in enumerate(p[::-1] * 100): ax.text(v + 1, i, f"{v:.0f}%", va="center")
    ax.set_xlim(0, 110); ax.set_title(title, fontsize=12); ax.spines[["top", "right"]].set_visible(False); plt.show()
show_row([load_photo(p) / 255 for p in DEFAULTS.values()][:3], list(DEFAULTS)[:3])
show_row([load_photo(p) / 255 for p in DEFAULTS.values()][3:], list(DEFAULTS)[3:])
UPLOADED = None

In [ ]:
#@title (Optional) Upload your own photo, then pick "My upload" in Step 1

from google.colab import files
up = files.upload()
if up:
    UPLOADED = list(up)[0]
    show_row([load_photo(UPLOADED) / 255], ["Your photo"])
    print("Note: this model knows only 4 tomato-leaf conditions. It will put ANY photo into one of them.")
    print("So its answer for your photo is not a diagnosis. Watch what the heatmap highlights instead.")

In [ ]:
#@title Step 1 · Pick a photo, then press ▶
assert "head" in globals(), "Run Step 0 first (the Setup cell at the top) and wait for: Model ready."
PHOTO = "Field 1 (healthy)" #@param ["Lab 1 (early blight)", "Lab 2 (healthy)", "Lab 3 (septoria)", "Field 1 (healthy)", "Field 2 (early blight)", "Field 3 (septoria)", "My upload"]
if PHOTO == "My upload":
    assert UPLOADED, "Upload a photo first (the optional cell above), or pick another photo."
    path, true = UPLOADED, None
else:
    path = DEFAULTS[PHOTO]; true = NAMES[CLASSES.index(path.split("/")[-2])]
img = load_photo(path)
p = predict(img); cam, c = gradcam(img)
show_row([img / 255, overlay(img, cam)],
         [f"True label: {true}" if true else "Your photo (no true label)",
          f"Model says: {NAMES[c]} ({p[c]:.0%})"],
         ["black", "black" if (true is None or NAMES[c] == true) else "#a11"])
bars(p, "The model's scores for all four conditions")
print("Red = the regions Grad-CAM links most strongly to the answer above. Blue = weakly.")
print("Is the red on the leaf, on the spots, or somewhere else?")

**Look before you go on:** where is the red? On the diseased spots, on the whole leaf, on the background, or on something else in the photo?

A heatmap is a **clue**, not proof. Step 2 checks the clue.

In [ ]:
#@title Step 2 · Grey out part of the photo and ask again, then press ▶
HIDE = "the red region (hottest part of the heatmap)" #@param ["the red region (hottest part of the heatmap)", "everything except the red region", "only the outer border", "a box I choose (use the sliders)"]
BOX_X = 50 #@param {type:"slider", min:0, max:100, step:5}
BOX_Y = 50 #@param {type:"slider", min:0, max:100, step:5}
BOX_SIZE = 30 #@param {type:"slider", min:10, max:80, step:5}
hot = cam >= np.quantile(cam, 0.80)          # the hottest 20% of the image
mask = np.zeros((IMG, IMG), bool)
if HIDE.startswith("the red"): mask = hot
elif HIDE.startswith("everything"): mask = ~hot
elif HIDE.startswith("only the outer"): mask[:16] = mask[-16:] = True; mask[:, :16] = mask[:, -16:] = True
else:
    s = int(IMG * BOX_SIZE / 100); cx = int(IMG * BOX_X / 100); cy = int(IMG * BOX_Y / 100)
    mask[max(0, cy - s // 2):cy + s // 2, max(0, cx - s // 2):cx + s // 2] = True
masked = img.copy(); masked[mask] = 128.0     # plain grey
p2 = predict(masked); cam2, c2 = gradcam(masked)
show_row([overlay(img, cam), masked / 255, overlay(masked, cam2)],
         [f"Before: {NAMES[c]} ({p[c]:.0%})", "What the model sees now", f"After: {NAMES[c2]} ({p2[c2]:.0%})"])
if c2 != c:
    print(f"The answer CHANGED: {NAMES[c]} -> {NAMES[c2]}. The hidden part mattered to the model.")
else:
    print(f"The answer stayed {NAMES[c]}; its score went from {p[c]:.0%} to {p2[c]:.0%}.")
print("Try another option, or another photo in Step 1.")

In [ ]:
#@title Step 3 · Save my result: type your name, then press ▶
assert "p2" in globals(), "Run Step 1 and Step 2 first."
MY_NAME = "" #@param {type:"string"}
from matplotlib import font_manager
serif = [f for f in ["Times New Roman", "Liberation Serif", "DejaVu Serif"] if any(f == x.name for x in font_manager.fontManager.ttflist)]
plt.rcParams.update({"font.family": "serif", "font.serif": serif or ["serif"]})
what = {"the red": "the red region", "everything": "everything except the red region",
        "only the outer": "the outer border", "a box": "a box I chose"}[next(k for k in ["the red", "everything", "only the outer", "a box"] if HIDE.startswith(k))]
if c2 != c:
    verdict = f"Hiding {what} CHANGED the answer: {NAMES[c]} → {NAMES[c2]}. The hidden part mattered to the model."
else:
    verdict = f"Hiding {what} did not change the answer ({NAMES[c]}: {p[c]:.0%} → {p2[c]:.0%})."
fig, axs = plt.subplots(1, 4, figsize=(16, 5.6))
panels = [(img / 255, "1. The photo" + (f"\n(true: {true})" if true else "\n(my own photo)")),
          (overlay(img, cam), f"2. Model says {NAMES[c]} ({p[c]:.0%})\nRed = where it looked"),
          (masked / 255, f"3. I hid {what}"),
          (overlay(masked, cam2), f"4. Now it says {NAMES[c2]} ({p2[c2]:.0%})")]
for ax, (im, t) in zip(axs, panels):
    ax.imshow(im); ax.set_title(t, fontsize=14); ax.axis("off")
title = "My Grad-CAM test" + (f" · {MY_NAME.strip()}" if MY_NAME.strip() else "")
fig.suptitle(title, fontsize=20, fontweight="bold", y=0.98)
fig.text(0.5, 0.075, verdict, ha="center", fontsize=14)
fig.text(0.5, 0.025, "A heatmap is a clue, not proof. Hiding a region tests the clue.   ·   AICTE QIP · IIIT Kottayam · FACTS-H Lab",
         ha="center", fontsize=11, color="#555")
plt.subplots_adjust(left=0.02, right=0.98, top=0.80, bottom=0.14, wspace=0.06)
fig.savefig("my_gradcam_result.png", dpi=150, facecolor="white")
plt.show()
try:
    from google.colab import files; files.download("my_gradcam_result.png")
    print("Saved: my_gradcam_result.png is downloading to your computer.")
except Exception:
    print("Saved as my_gradcam_result.png (Files panel on the left).")


## What to type in the chat
1. Which photo and which option did you try?
2. Did the answer change when you greyed out the red region?
3. One sentence: what does that tell you about the heatmap?